Apriori Algorithm

In [ ]:
from itertools import combinations

transactions = [
    ['I1', 'I2', 'I5'], ['I2', 'I4'], ['I2', 'I3'],
    ['I1', 'I2', 'I4'], ['I1', 'I3'], ['I2', 'I3'],
    ['I1', 'I3'], ['I1', 'I2', 'I3', 'I5'], ['I1', 'I2', 'I3']
]

min_sup = 2
min_confidence = 2
N=len(transactions)

In [ ]:
#generate frequent items
def get_frequent_itemsets(transactions,min_sup):
  frequent_items = {}

  counts={}
  for transaction in transactions:
    for item in transaction:
      item = frozenset([item])
      if item not in counts.keys():
        counts[item]=1
      else:
        counts[item]+=1

  current_L = {k:v for k,v in counts.items() if v >= min_sup}
  frequent_items.update(current_L)

  k=2
  while current_L:
    unique_items = set()
    for itemsets in current_L:
      for item in itemsets:
        unique_items.add(item)

    candidates = set(frozenset(c) for c in combinations(unique_items,k))

    current_counts = {}
    for cand in candidates:
      count=0
      for transaction in transactions:
        if cand.issubset(transaction):
          count+=1
      if count>= min_sup :
        current_counts[cand] = count

    if current_counts:
      frequent_items.update(current_counts)
      current_L = current_counts
      k+=1
    else:
      break
  return frequent_items


In [ ]:
support_map = get_frequent_itemsets(transactions,1)
print(support_map)

{frozenset({'I1'}): 6, frozenset({'I2'}): 7, frozenset({'I5'}): 2, frozenset({'I4'}): 2, frozenset({'I3'}): 6, frozenset({'I2', 'I4'}): 2, frozenset({'I2', 'I1'}): 4, frozenset({'I1', 'I4'}): 1, frozenset({'I1', 'I3'}): 4, frozenset({'I2', 'I5'}): 2, frozenset({'I2', 'I3'}): 4, frozenset({'I3', 'I5'}): 1, frozenset({'I1', 'I5'}): 2, frozenset({'I1', 'I3', 'I5'}): 1, frozenset({'I2', 'I1', 'I5'}): 2, frozenset({'I2', 'I3', 'I5'}): 1, frozenset({'I2', 'I1', 'I4'}): 1, frozenset({'I2', 'I1', 'I3'}): 2, frozenset({'I2', 'I1', 'I3', 'I5'}): 1}


In [ ]:
#generate association rules
def generate_association_rules(frequent_itemsets,min_confidence):
  for itemset in frequent_itemsets:
    k = len(itemset)
    if k<2:
      continue

    for i in range(1,k):
      for antecedents in combinations(itemset,i):
        antecedent = frozenset(antecedents)
        consequent = itemset - antecedent

        support_AB = frequent_itemsets[itemset]
        support_A = frequent_itemsets[antecedent]
        support_B = frequent_itemsets[consequent]

        if (support_AB/support_A) >= min_confidence:
          print( f"{set(antecedent)} -> {set(consequent)}")

In [ ]:
print("Association Rules Generated:")
generate_association_rules(support_map,0.4)

Association Rules Generated:
{'I4'} -> {'I2'}
{'I2'} -> {'I1'}
{'I1'} -> {'I2'}
{'I4'} -> {'I1'}
{'I1'} -> {'I3'}
{'I3'} -> {'I1'}
{'I5'} -> {'I2'}
{'I2'} -> {'I3'}
{'I3'} -> {'I2'}
{'I5'} -> {'I3'}
{'I5'} -> {'I1'}
{'I5'} -> {'I1', 'I3'}
{'I1', 'I5'} -> {'I3'}
{'I3', 'I5'} -> {'I1'}
{'I5'} -> {'I2', 'I1'}
{'I2', 'I1'} -> {'I5'}
{'I2', 'I5'} -> {'I1'}
{'I1', 'I5'} -> {'I2'}
{'I5'} -> {'I2', 'I3'}
{'I2', 'I5'} -> {'I3'}
{'I3', 'I5'} -> {'I2'}
{'I4'} -> {'I2', 'I1'}
{'I2', 'I4'} -> {'I1'}
{'I1', 'I4'} -> {'I2'}
{'I2', 'I1'} -> {'I3'}
{'I2', 'I3'} -> {'I1'}
{'I1', 'I3'} -> {'I2'}
{'I5'} -> {'I2', 'I1', 'I3'}
{'I2', 'I5'} -> {'I1', 'I3'}
{'I1', 'I5'} -> {'I2', 'I3'}
{'I3', 'I5'} -> {'I2', 'I1'}
{'I2', 'I1', 'I3'} -> {'I5'}
{'I2', 'I1', 'I5'} -> {'I3'}
{'I2', 'I3', 'I5'} -> {'I1'}
{'I1', 'I3', 'I5'} -> {'I2'}


FP Tree Growth

In [ ]:
# 1. SETUP DATA
dataset = [
    ['Milk', 'Bread', 'Beer'],
    ['Bread', 'Diaper', 'Eggs'],
    ['Milk', 'Diaper', 'Beer', 'Cola'],
    ['Bread', 'Milk', 'Diaper', 'Beer'],
    ['Bread', 'Milk', 'Diaper', 'Cola']
]
min_sup = 2

In [ ]:
freq_items = {}

def fp_growth(transactions,suffix,min_sup):
  counts={}
  for t in transactions:
    for item in t:
      if item not in counts.keys():
        counts[item]=1
      else:
        counts[item]+=1
  valid_items = {k:v for k,v in counts.items() if v>= min_sup}

  if not valid_items:
    return

  for item,count in valid_items.items():
    new_itemset = suffix + [item]
    freq_items[tuple(sorted(new_itemset))] = count

  sorted_items = sorted(valid_items.keys(),key = lambda k : valid_items[k])

  for item in sorted_items:
    cond_dataset=[]

    for t in transactions:
      if item in t:
        path = [x for x in t if x in valid_items and x!=item]
        path.sort(key = lambda k :valid_items[k],reverse=True)
        if path:
          cond_dataset.append(path)
    if cond_dataset:
      fp_growth(cond_dataset,suffix+[item],min_sup)

fp_growth(dataset, [], 2)

print("--- FP-Growth Results ---")
for itemset, support in freq_items.items():
    print(f"{list(itemset)}: {support}")


--- FP-Growth Results ---
['Milk']: 4
['Bread']: 4
['Beer']: 3
['Diaper']: 4
['Cola']: 2
['Cola', 'Milk']: 2
['Cola', 'Diaper']: 2
['Cola', 'Diaper', 'Milk']: 2
['Beer', 'Milk']: 3
['Beer', 'Bread']: 2
['Beer', 'Diaper']: 2
['Beer', 'Bread', 'Milk']: 2
['Beer', 'Diaper', 'Milk']: 2
['Bread', 'Milk']: 3
['Diaper', 'Milk']: 3
['Bread', 'Diaper', 'Milk']: 2
['Bread', 'Diaper']: 3
